# P1 - Predicción de Subscripción a un Producto Bancario
**Asignatura:** Aprendizaje Automático — UC3M 2025-26

**Miembros del grupo:**
- Nombre Apellidos (NIA: 100522196)

---

## 0. Imports y configuración general

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn import metrics
import joblib
import warnings
warnings.filterwarnings('ignore')

NIA  = 100522196
SEED = NIA % (2**32)
np.random.seed(SEED)
print(f'Semilla usada: {SEED}')

## 1. Carga de datos

Fórmula: $xx = a + b$, NIA `100522196` → `a=9`, `b=6` → `xx=15`

In [ ]:
last_two = NIA % 100
a  = last_two // 10
b  = last_two % 10
xx = a + b
print(f'NIA: {NIA}  →  a={a}, b={b}  →  xx={xx}')

df             = pd.read_pickle(f'bank_{xx}.pkl')
df_competition = pd.read_pickle(f'bank_competition.pkl')
print('Datos principales:  ', df.shape)
print('Datos competición:  ', df_competition.shape)

## 2. EDA Simplificado (0,3 puntos)

### 2.1 Dimensiones y tipos de variables

In [ ]:
TARGET = 'deposit'
print(f'Instancias : {df.shape[0]}')
print(f'Variables  : {df.shape[1]} (incluyendo target)\n')

num_features = df.drop(columns=TARGET).select_dtypes(include=['int64','float64']).columns.tolist()
cat_features = df.drop(columns=TARGET).select_dtypes(include=['object']).columns.tolist()

print(f'Variables numéricas   ({len(num_features)}): {num_features}')
print(f'Variables categóricas ({len(cat_features)}): {cat_features}')

summary = pd.DataFrame({
    'dtype'    : df.drop(columns=TARGET).dtypes,
    'tipo'     : ['numérica' if str(d) in ['int64','float64'] else 'categórica'
                  for d in df.drop(columns=TARGET).dtypes],
    'n_unicos' : df.drop(columns=TARGET).nunique(),
    'ejemplo'  : [df[c].dropna().iloc[0] for c in df.drop(columns=TARGET).columns]
})
display(summary)

### 2.2 Valores faltantes

In [ ]:
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'valores_faltantes': missing, 'porcentaje (%)': missing_pct}).query('valores_faltantes > 0')
if missing_df.empty:
    print('No hay valores faltantes.')
else:
    display(missing_df)

### 2.3 Tipo de problema y desbalanceo

In [ ]:
counts = df[TARGET].value_counts()
props  = df[TARGET].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({'count': counts, '%': props}))
ratio = counts.max() / counts.min()
print(f'Ratio: {ratio:.2f} → {"BALANCEADO" if ratio < 1.5 else "DESBALANCEADO"}')

fig, ax = plt.subplots(figsize=(5, 3))
counts.plot(kind='bar', ax=ax, color=['#d9534f','#5cb85c'], edgecolor='white', width=0.5)
ax.set_title('Distribución de deposit')
ax.set_xticklabels(counts.index, rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())} ({p.get_height()/len(df)*100:.1f}%)',
                (p.get_x() + p.get_width()/2, p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.show()

### 2.4 Cardinalidad y columnas constantes/ID

In [ ]:
card_df = pd.DataFrame({
    'n_unicos'               : df[cat_features].nunique(),
    'alta_cardinalidad (>10)': df[cat_features].nunique() > 10,
    'valores'                : [list(df[c].dropna().unique()) for c in cat_features]
})
display(card_df)
const_cols = [c for c in df.columns if df[c].nunique() <= 1]
id_cols    = [c for c in df.columns if df[c].nunique() == len(df)]
print(f'Columnas constantes: {const_cols if const_cols else "Ninguna"}')
print(f'Posibles columnas ID: {id_cols if id_cols else "Ninguna"}')

### 2.5 Análisis y preprocesamiento de `pdays`

In [ ]:
n_minus1  = (df['pdays'] == -1).sum()
n_contact = (df['pdays'] >= 0).sum()
print(f'Valor -1 (sin contacto): {n_minus1} ({n_minus1/len(df)*100:.1f}%)')
print(f'Valor >= 0 (contactado): {n_contact} ({n_contact/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(df['pdays'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('pdays (completo)')
axes[1].hist(df.loc[df['pdays'] >= 0, 'pdays'], bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('pdays (solo contactados)')
plt.tight_layout()
plt.show()

In [ ]:
def preprocess_pdays(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data['contacted_before'] = (data['pdays'] != -1).astype(int)
    data['pdays_clean']      = data['pdays'].clip(lower=0)
    return data.drop(columns=['pdays'])

df             = preprocess_pdays(df)
df_competition = preprocess_pdays(df_competition)

num_features = df.drop(columns=TARGET).select_dtypes(include=['int64','float64']).columns.tolist()
cat_features = df.drop(columns=TARGET).select_dtypes(include=['object']).columns.tolist()
print(f'Shape actualizado: {df.shape}')
print(f'Numéricas  ({len(num_features)}): {num_features}')
print(f'Categóricas ({len(cat_features)}): {cat_features}')

## 3. Configuración de la evaluación (0,1 puntos)

- **Métrica:** `accuracy` — válida porque el dataset está balanceado (ratio 1.11).
- **Outer:** Holdout train (2/3) / test (1/3) con estratificación.
- **Inner:** 3-Fold Cross Validation sobre train.

> Test **no se usa** hasta elegir el mejor modelo.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

#codificar el target como 0/1 para compatibilidad con sklearn
le = LabelEncoder()
y_enc = le.fit_transform(y)   # 'no' -> 0, 'yes' -> 1
print(f'Clases originales: {le.classes_}  →  codificadas: {le.transform(le.classes_)}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=1/3, random_state=SEED, stratify=y_enc
)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=SEED)

print(f'\nTrain: {X_train.shape[0]} instancias | Test: {X_test.shape[0]} instancias')
display(pd.DataFrame({
    'Train %': pd.Series(y_train).value_counts(normalize=True).mul(100).round(1).rename({0:'no',1:'yes'}),
    'Test %' : pd.Series(y_test).value_counts(normalize=True).mul(100).round(1).rename({0:'no',1:'yes'})
}))

## 4. Métodos básicos: KNN y Trees (1,0 punto)

### 4.1 Selección del escalador con KNN (hiperparámetros por omisión)

In [ ]:
def make_preprocessor(scaler):
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore'))
    ])
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler',  scaler)
    ])
    return ColumnTransformer([
        ('cat', cat_pipe, cat_features),
        ('num', num_pipe, num_features)
    ])

In [ ]:
scalers = {'MinMax': MinMaxScaler(), 'Standard': StandardScaler(), 'Robust': RobustScaler()}
scaler_results = []

for name, scaler in scalers.items():
    pipe   = Pipeline([('preprocessor', make_preprocessor(scaler)), ('knn', KNeighborsClassifier())])
    t0     = time.time()
    scores = cross_val_score(pipe, X_train, y_train, scoring='accuracy', cv=inner_cv)
    t1     = time.time()
    scaler_results.append({'Scaler': name, 'CV Accuracy': round(scores.mean(),4),
                           'Std': round(scores.std(),4), 'Tiempo (s)': round(t1-t0,2)})

scaler_df = pd.DataFrame(scaler_results).set_index('Scaler')
display(scaler_df)
best_scaler_name = scaler_df['CV Accuracy'].idxmax()
print(f'\n→ Mejor escalador: {best_scaler_name} (accuracy = {scaler_df.loc[best_scaler_name, "CV Accuracy"]})')

In [ ]:
SCALER = StandardScaler()
print('Escalador seleccionado: StandardScaler — se usará en KNN, SVM y modelos lineales.')

### 4.2 KNN con hiperparámetros por omisión y comparación con modelo trivial

In [ ]:
# KNN por defecto
pipe_knn_default = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('knn', KNeighborsClassifier())
])
t0 = time.time()
scores_knn = cross_val_score(pipe_knn_default, X_train, y_train, scoring='accuracy', cv=inner_cv)
t1 = time.time()
tiempo_knn_default = round(t1 - t0, 2)

#modelo trivial Dummy
dummy = DummyClassifier(strategy='most_frequent')
scores_dummy = cross_val_score(dummy, X_train, y_train, scoring='accuracy', cv=inner_cv)

resultados_default = pd.DataFrame([
    {'Modelo': 'Dummy (most_frequent)', 'CV Accuracy': round(scores_dummy.mean(),4),
     'Std': round(scores_dummy.std(),4), 'Tiempo (s)': '-'},
    {'Modelo': 'KNN (default HP)',      'CV Accuracy': round(scores_knn.mean(),4),
     'Std': round(scores_knn.std(),4),  'Tiempo (s)': tiempo_knn_default},
]).set_index('Modelo')
display(resultados_default)
print(f'\n→ KNN supera al modelo trivial en {scores_knn.mean() - scores_dummy.mean():.4f} puntos de accuracy.')

### 4.3 HPO de KNN con GridSearchCV

Se buscan los mejores valores de `n_neighbors`, `weights` y `p` (distancia Manhattan vs Euclidea).

In [ ]:
param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 11, 15, 21, 31, 51],
    'knn__weights'    : ['uniform', 'distance'],
    'knn__p'          : [1, 2]
}

pipe_knn_hpo = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('knn', KNeighborsClassifier())
])

search_knn = GridSearchCV(
    pipe_knn_hpo, param_grid_knn,
    scoring='accuracy', cv=inner_cv, n_jobs=-1
)

t0 = time.time()
np.random.seed(SEED)
search_knn.fit(X_train, y_train)
t1 = time.time()
tiempo_knn_hpo = round(t1 - t0, 2)

print(f'Mejores hiperparámetros : {search_knn.best_params_}')
print(f'Mejor accuracy (inner)  : {search_knn.best_score_:.4f}')
print(f'Tiempo HPO              : {tiempo_knn_hpo}s')

In [ ]:
#tabla resumen KNN: default vs HPO
resumen_knn = pd.DataFrame([
    {'Modelo': 'Dummy (most_frequent)', 'CV Accuracy': round(scores_dummy.mean(),4), 'Tiempo (s)': '-'},
    {'Modelo': 'KNN (default HP)',      'CV Accuracy': round(scores_knn.mean(),4),   'Tiempo (s)': tiempo_knn_default},
    {'Modelo': 'KNN (HPO)',             'CV Accuracy': round(search_knn.best_score_,4), 'Tiempo (s)': tiempo_knn_hpo},
]).set_index('Modelo')
display(resumen_knn)

mejora = search_knn.best_score_ - scores_knn.mean()
print(f'\n→ Mejora del HPO sobre default: +{mejora:.4f} accuracy')
print(f'→ Coste del HPO: {tiempo_knn_hpo}s vs {tiempo_knn_default}s (x{tiempo_knn_hpo/tiempo_knn_default:.0f} más lento)')

### 4.4 Efecto de `n_neighbors` en la accuracy

In [ ]:
ks   = [1, 3, 5, 7, 11, 15, 21, 31, 51, 75, 101]
accs = []

for k in ks:
    pipe_k = Pipeline([
        ('preprocessor', make_preprocessor(StandardScaler())),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    s = cross_val_score(pipe_k, X_train, y_train, scoring='accuracy', cv=inner_cv)
    accs.append(s.mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, accs, marker='o', color='steelblue', linewidth=2)
ax.axhline(scores_dummy.mean(), color='red', linestyle='--', label='Dummy baseline')
ax.axvline(search_knn.best_params_['knn__n_neighbors'], color='green',
           linestyle=':', label=f'Mejor k (HPO) = {search_knn.best_params_["knn__n_neighbors"]}')
ax.set_xlabel('n_neighbors (k)')
ax.set_ylabel('CV Accuracy (3-Fold)')
ax.set_title('Efecto de n_neighbors en KNN')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Observación: valores muy bajos de k (k=1) producen overfitting (alta varianza).')
print('Valores muy altos underfitting (el modelo se vuelve demasiado suave).')
print(f'El rango óptimo está entre k=7 y k=15 para este problema.')

### 4.5 Trees — default HP, árbol interpretable y HPO

In [ ]:
# Tree con hiperparámetros por omisión
# Los árboles no necesitan escalado
def make_preprocessor_tree():
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore'))
    ])
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean'))
    ])
    return ColumnTransformer([
        ('cat', cat_pipe, cat_features),
        ('num', num_pipe, num_features)
    ])

pipe_tree_default = Pipeline([
    ('preprocessor', make_preprocessor_tree()),
    ('tree', DecisionTreeClassifier(random_state=SEED))
])
t0 = time.time()
scores_tree = cross_val_score(pipe_tree_default, X_train, y_train, scoring='accuracy', cv=inner_cv)
t1 = time.time()
tiempo_tree_default = round(t1 - t0, 2)

resumen_tree_default = pd.DataFrame([
    {'Modelo': 'Dummy (most_frequent)', 'CV Accuracy': round(scores_dummy.mean(), 4), 'Tiempo (s)': '-'},
    {'Modelo': 'KNN (default HP)',      'CV Accuracy': round(scores_knn.mean(), 4),   'Tiempo (s)': tiempo_knn_default},
    {'Modelo': 'Tree (default HP)',     'CV Accuracy': round(scores_tree.mean(), 4),  'Tiempo (s)': tiempo_tree_default},
]).set_index('Modelo')
display(resumen_tree_default)

### 4.6 Árbol poco profundo

Un árbol con `max_depth=3` permite entender qué variables usa el modelo para tomar decisiones.
Para visualizarlo se transforma primero el conjunto de train con el preprocesador.

In [ ]:
# Transformar X_train para obtener nombres de features tras OHE
preprocessor_tree_fit = make_preprocessor_tree()
X_train_transformed   = preprocessor_tree_fit.fit_transform(X_train, y_train)
feat_names_tree       = list(preprocessor_tree_fit.get_feature_names_out())

tree_shallow = DecisionTreeClassifier(max_depth=3, random_state=SEED)
tree_shallow.fit(X_train_transformed, y_train)

print('Reglas del árbol (max_depth=3):')
print(export_text(tree_shallow, feature_names=feat_names_tree))

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(
    tree_shallow,
    feature_names=feat_names_tree,
    class_names=['no', 'yes'],
    filled=True,
    rounded=True,
    ax=ax
)
ax.set_title('Árbol de decisión (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

print('Variables más relevantes en las primeras decisiones:')
print('  - duration: duración del último contacto (segundos)')
print('  - poutcome_success: éxito en campañas anteriores')
print('  - contact_unknown: tipo de contacto desconocido')

### 4.7 Efecto de `max_depth` en la accuracy

In [ ]:
depths = [1, 2, 3, 4, 5, 7, 10, None]
accs_depth = []

for d in depths:
    pipe_d = Pipeline([
        ('preprocessor', make_preprocessor_tree()),
        ('tree', DecisionTreeClassifier(max_depth=d, random_state=SEED))
    ])
    s = cross_val_score(pipe_d, X_train, y_train, scoring='accuracy', cv=inner_cv)
    accs_depth.append(s.mean())

depth_labels = [str(d) if d is not None else 'None' for d in depths]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depth_labels, accs_depth, marker='o', color='darkorange', linewidth=2)
ax.axhline(scores_dummy.mean(), color='red', linestyle='--', label='Dummy baseline')
ax.set_xlabel('max_depth')
ax.set_ylabel('CV Accuracy (3-Fold)')
ax.set_title('Efecto de max_depth en el árbol de decisión')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_depth_idx = accs_depth.index(max(accs_depth))
print(f'Mejor max_depth: {depth_labels[best_depth_idx]} (accuracy = {max(accs_depth):.4f})')
print('Observación: árboles muy profundos (None) sufren overfitting y caen en CV.')

### 4.8 HPO de árboles con GridSearchCV

In [ ]:
param_grid_tree = {
    'tree__max_depth'        : [3, 5, 7, 10, 15, 20],
    'tree__min_samples_split': [2, 10, 20, 50],
    'tree__min_samples_leaf' : [1, 5, 10]
}
pipe_tree_hpo = Pipeline([
    ('preprocessor', make_preprocessor_tree()),
    ('tree', DecisionTreeClassifier(random_state=SEED))
])
search_tree = GridSearchCV(pipe_tree_hpo, param_grid_tree, scoring='accuracy', cv=inner_cv, n_jobs=-1)
t0 = time.time()
search_tree.fit(X_train, y_train)
t1 = time.time()
tiempo_tree_hpo = round(t1 - t0, 1)

print(f'Mejores parámetros: {search_tree.best_params_}')
print(f'Mejor accuracy (inner): {search_tree.best_score_:.4f}')
print(f'Tiempo HPO: {tiempo_tree_hpo}s')

### 4.9 Conclusiones — Métodos básicos (KNN y Trees)

In [ ]:
resumen_basicos = pd.DataFrame([
    {'Modelo': 'Dummy',        'CV Accuracy': round(scores_dummy.mean(),4), 'Tiempo (s)': '-',               'HPO': '-'},
    {'Modelo': 'KNN default',  'CV Accuracy': round(scores_knn.mean(),4),   'Tiempo (s)': tiempo_knn_default,'HPO': 'No'},
    {'Modelo': 'KNN HPO',      'CV Accuracy': 0.8057,                        'Tiempo (s)': '33.4',           'HPO': 'Sí'},
    {'Modelo': 'Tree default', 'CV Accuracy': round(scores_tree.mean(),4),   'Tiempo (s)': tiempo_tree_default,'HPO': 'No'},
    {'Modelo': 'Tree HPO',     'CV Accuracy': round(search_tree.best_score_,4),'Tiempo (s)': tiempo_tree_hpo, 'HPO': 'Sí'},
]).set_index('Modelo')
display(resumen_basicos)

**Conclusiones:**

- Ambos métodos superan claramente al modelo trivial (Dummy: 0.5254).
- El HPO mejora los árboles significativamente (+0.04 sobre default), pero con coste alto (~32s).
- El HPO de KNN mejora poco (+0.005) y también es costoso (~33s).
- Los árboles con HPO son más rápidos de entrenar que KNN y obtienen mejor resultado que KNN sin HPO.
- El mejor método básico es **Tree HPO** (0.8313), aunque KNN HPO queda cerca (0.8057).

## 5. Métodos avanzados: Modelos Lineales y SVMs (0,6 puntos)

Se usan los mismos pipelines con `StandardScaler`. Los modelos lineales y SVM requieren escalado.

### 5.1 Modelos lineales y SVM con hiperparámetros por omisión

In [ ]:
# Regresion Logística L2 (por defecto)
pipe_lr = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('lr', LogisticRegression(max_iter=1000, random_state=SEED))
])
t0 = time.time()
scores_lr = cross_val_score(pipe_lr, X_train, y_train, scoring='accuracy', cv=inner_cv)
t1 = time.time(); tiempo_lr = round(t1-t0,2)

# Regresion Logística L1
pipe_lr_l1 = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('lr', LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=SEED))
])
t0 = time.time()
scores_lr_l1 = cross_val_score(pipe_lr_l1, X_train, y_train, scoring='accuracy', cv=inner_cv)
t1 = time.time(); tiempo_lr_l1 = round(t1-t0,2)

# SVM
pipe_svm = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('svm', SVC(random_state=SEED))
])
t0 = time.time()
scores_svm = cross_val_score(pipe_svm, X_train, y_train, scoring='accuracy', cv=inner_cv)
t1 = time.time(); tiempo_svm = round(t1-t0,2)

resumen_avanzados_default = pd.DataFrame([
    {'Modelo': 'LR L2 (default)', 'CV Accuracy': round(scores_lr.mean(),4),    'Std': round(scores_lr.std(),4),    'Tiempo (s)': tiempo_lr},
    {'Modelo': 'LR L1 (default)', 'CV Accuracy': round(scores_lr_l1.mean(),4), 'Std': round(scores_lr_l1.std(),4), 'Tiempo (s)': tiempo_lr_l1},
    {'Modelo': 'SVM (default)',   'CV Accuracy': round(scores_svm.mean(),4),    'Std': round(scores_svm.std(),4),   'Tiempo (s)': tiempo_svm},
]).set_index('Modelo')
display(resumen_avanzados_default)

### 5.2 HPO de modelos lineales y SVM


In [ ]:
# HPO Regresión Logística L2
param_grid_lr = {'lr__C': [0.001, 0.01, 0.1, 1, 10, 100]}
pipe_lr_hpo = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('lr', LogisticRegression(max_iter=1000, random_state=SEED))
])
search_lr = GridSearchCV(pipe_lr_hpo, param_grid_lr, scoring='accuracy', cv=inner_cv, n_jobs=-1)
t0 = time.time(); search_lr.fit(X_train, y_train); t1 = time.time()
tiempo_lr_hpo = round(t1-t0,2)

# HPO Regresión Logística L1
param_grid_lr_l1 = {'lr__C': [0.001, 0.01, 0.1, 1, 10, 100]}
pipe_lr_l1_hpo = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('lr', LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=SEED))
])
search_lr_l1 = GridSearchCV(pipe_lr_l1_hpo, param_grid_lr_l1, scoring='accuracy', cv=inner_cv, n_jobs=-1)
t0 = time.time(); search_lr_l1.fit(X_train, y_train); t1 = time.time()
tiempo_lr_l1_hpo = round(t1-t0,2)

# HPO SVM
param_grid_svm = {'svm__C': [0.1, 1, 10], 'svm__kernel': ['linear', 'rbf']}
pipe_svm_hpo = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('svm', SVC(random_state=SEED))
])
search_svm = GridSearchCV(pipe_svm_hpo, param_grid_svm, scoring='accuracy', cv=inner_cv, n_jobs=-1)
t0 = time.time(); search_svm.fit(X_train, y_train); t1 = time.time()
tiempo_svm_hpo = round(t1-t0,1)

resumen_avanzados_hpo = pd.DataFrame([
    {'Modelo': 'LR L2 HPO', 'CV Accuracy': round(search_lr.best_score_,4),     'Mejores params': str(search_lr.best_params_),     'Tiempo (s)': tiempo_lr_hpo},
    {'Modelo': 'LR L1 HPO', 'CV Accuracy': round(search_lr_l1.best_score_,4),  'Mejores params': str(search_lr_l1.best_params_),  'Tiempo (s)': tiempo_lr_l1_hpo},
    {'Modelo': 'SVM HPO',   'CV Accuracy': round(search_svm.best_score_,4),    'Mejores params': str(search_svm.best_params_),    'Tiempo (s)': tiempo_svm_hpo},
]).set_index('Modelo')
display(resumen_avanzados_hpo)

### 5.3 Importancia de atributos — Regresión Logística L1

La regularización L1 lleva a cero los coeficientes de las variables menos relevantes. 
Los coeficientes no nulos indican las variables más importantes para el modelo.

In [ ]:
# Reentrenar LR L1 con mejores params para extraer coeficientes
search_lr_l1.best_estimator_.fit(X_train, y_train)
lr_fitted  = search_lr_l1.best_estimator_.named_steps['lr']
pre_fitted = search_lr_l1.best_estimator_.named_steps['preprocessor']
feat_names = list(pre_fitted.get_feature_names_out())

coef_series = pd.Series(lr_fitted.coef_[0], index=feat_names)
coef_abs    = coef_series.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
coef_abs.plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 15 atributos más relevantes (|coef| LR L1)')
ax.set_xlabel('|Coeficiente|')
plt.tight_layout()
plt.show()

n_zero = (lr_fitted.coef_[0] == 0).sum()
print(f'Variables con coeficiente = 0 (L1 las eliminó): {n_zero} de {len(feat_names)}')
print(f'\nTop 5 variables más relevantes:')
for feat, val in coef_abs.head(5).items():
    print(f'  {feat:<35} {val:.3f}')

## 6. Resultados y modelo final (0,3 puntos)

### 6.1 Tabla comparativa global y selección del mejor modelo


In [ ]:
tabla_final = pd.DataFrame([
    {'Modelo': 'Dummy',          'CV Accuracy (inner)': round(scores_dummy.mean(),4)},
    {'Modelo': 'KNN default',    'CV Accuracy (inner)': round(scores_knn.mean(),4)},
    {'Modelo': 'KNN HPO',        'CV Accuracy (inner)': 0.8057},
    {'Modelo': 'Tree default',   'CV Accuracy (inner)': round(scores_tree.mean(),4)},
    {'Modelo': 'Tree HPO',       'CV Accuracy (inner)': round(search_tree.best_score_,4)},
    {'Modelo': 'LR L2 default',  'CV Accuracy (inner)': round(scores_lr.mean(),4)},
    {'Modelo': 'LR L1 default',  'CV Accuracy (inner)': round(scores_lr_l1.mean(),4)},
    {'Modelo': 'LR L2 HPO',      'CV Accuracy (inner)': round(search_lr.best_score_,4)},
    {'Modelo': 'LR L1 HPO',      'CV Accuracy (inner)': round(search_lr_l1.best_score_,4)},
    {'Modelo': 'SVM default',    'CV Accuracy (inner)': round(scores_svm.mean(),4)},
    {'Modelo': 'SVM HPO',        'CV Accuracy (inner)': round(search_svm.best_score_,4)},
]).set_index('Modelo').sort_values('CV Accuracy (inner)', ascending=False)
display(tabla_final)
print(f'\n→ Mejor modelo (inner): {tabla_final.index[0]} con accuracy = {tabla_final.iloc[0,0]}')

### 6.2 Evaluación outer — rendimiento futuro en test

> Se usa el conjunto de test **por primera y única vez** para estimar el rendimiento futuro del mejor modelo.

In [ ]:
# Mejor modelo: SVM HPO (ya entrenado sobre X_train en la búsqueda)
y_pred_test = search_svm.predict(X_test)
acc_test    = metrics.accuracy_score(y_test, y_pred_test)
print(f'Accuracy en test (outer): {acc_test:.4f}')
print(f'Accuracy en inner (CV):   {search_svm.best_score_:.4f}')
print(f'Diferencia inner-outer:   {search_svm.best_score_ - acc_test:.4f}')

In [ ]:
# Matriz de confusión
cm = metrics.confusion_matrix(y_test, y_pred_test)
fig, ax = plt.subplots(figsize=(5, 4))
disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de confusión — SVM HPO (test)\nAccuracy = {acc_test:.4f}')
plt.tight_layout()
plt.show()
print(metrics.classification_report(y_test, y_pred_test, target_names=le.classes_))

### 6.3 Entrenamiento del modelo final

El modelo final se entrena con **todos los datos disponibles** (train + test).

In [ ]:
X_all = df.drop(columns=[TARGET])
y_all = le.transform(df[TARGET])

# Pipeline con los mejores hiperparámetros del SVM
final_pipe = Pipeline([
    ('preprocessor', make_preprocessor(StandardScaler())),
    ('svm', SVC(C=1, kernel='rbf', random_state=SEED))
])
np.random.seed(SEED)
final_pipe.fit(X_all, y_all)

joblib.dump(final_pipe, 'modelo_final.joblib')
print('Modelo final guardado en: modelo_final.joblib')
print(f'Entrenado con {X_all.shape[0]} instancias y {X_all.shape[1]} variables.')

### 6.4 Predicciones sobre datos de competición


In [ ]:
X_comp = df_competition.copy()

y_comp_pred     = final_pipe.predict(X_comp)
y_comp_labels   = le.inverse_transform(y_comp_pred)  # 0/1 -> 'no'/'yes'

predicciones = pd.DataFrame({'deposit': y_comp_labels})
predicciones.to_csv('predicciones.csv', index=False)

print('Predicciones guardadas en: predicciones.csv')
print(f'Total predicciones: {len(predicciones)}')
print(predicciones['deposit'].value_counts())

## 7. Tarea de elección abierta (0,4 puntos)

### Justificación

La variable `duration` (duración del último contacto en segundos) es la más importante del modelo, 
como muestra el árbol interpretable y la regresión logística L1. Sin embargo, **esta variable no se conoce 
antes de realizar la llamada**: solo se conoce una vez que la llamada ha terminado. Por tanto, un modelo que 
use `duration` no puede usarse para decidir **a quién llamar** antes de hacerlo.

La tarea consiste en:
1. Cuantificar el impacto de `duration` en el rendimiento.
2. Entrenar un modelo alternativo **sin `duration`** que sea más realista para uso en producción.
3. Comparar ambos modelos y discutir el trade-off.

In [ ]:
# Modelo con duration (ya entrenado en la búsqueda)
acc_con_duration = search_svm.best_score_
print(f'SVM HPO con duration (inner CV): {acc_con_duration:.4f}')

# Modelo sin duration
num_features_sin = [c for c in num_features if c != 'duration']

def make_preprocessor_sin_duration():
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore'))
    ])
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler',  StandardScaler())
    ])
    return ColumnTransformer([
        ('cat', cat_pipe, cat_features),
        ('num', num_pipe, num_features_sin)
    ])

pipe_sin_duration = Pipeline([
    ('preprocessor', make_preprocessor_sin_duration()),
    ('svm', SVC(C=1, kernel='rbf', random_state=SEED))
])
scores_sin = cross_val_score(pipe_sin_duration, X_train, y_train, scoring='accuracy', cv=inner_cv)
acc_sin_duration = scores_sin.mean()
print(f'SVM sin duration (inner CV):     {acc_sin_duration:.4f}')
print(f'Caída al eliminar duration:      {acc_con_duration - acc_sin_duration:.4f}')

In [ ]:
# Comparativa visual
fig, ax = plt.subplots(figsize=(6, 3))
modelos  = ['SVM con duration\n(no realista)', 'SVM sin duration\n(producción real)']
accs     = [acc_con_duration, acc_sin_duration]
colores  = ['steelblue', 'darkorange']
bars = ax.bar(modelos, accs, color=colores, edgecolor='white', width=0.4)
ax.set_ylim(0.65, 0.90)
ax.axhline(scores_dummy.mean(), color='red', linestyle='--', label='Dummy baseline')
ax.set_ylabel('CV Accuracy (inner)')
ax.set_title('Impacto de eliminar duration del modelo')
ax.legend()
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{acc:.4f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

print('Conclusión: eliminar duration reduce la accuracy en ~11 puntos.')
print('El modelo sin duration (0.7405) sigue siendo muy superior al Dummy (0.5254),')
print('y es el único viable para decidir a quién contactar antes de hacer la llamada.')

---
## Uso de IA Generativa
Durante el desarrollo de este notebook, se han utilizado herramientas de Inteligencia Artificial como asistencia técnica, con los siguientes propósitos:

Estructuración inicial: Creación de la base del proyecto y la definición de sus apartados principales.

Asistencia en la redacción: Uso de autocompletado al escribir código y formulación de prompts para mejorar la legibilidad y claridad de los comentarios explicativos al igual que la limpieza del código.

Resolución de problemas: Asistencia en la depuración de errores durante el desarrollo (como parámetros incorrectos) y consultas sobre la lógica o implementación de funciones desconocidas.

Mejora de estilo: Ayuda en el estilo de mystreamlit.py

Nota importante: Cabe recalcar que todo el código y los comentarios sugeridos o generados por la IA han sido rigurosamente revisados, analizados y validados de forma manual para asegurar su calidad y correcto funcionamiento en el proyecto.